<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        K-Means Clustering
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 10
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/02_KMeans_Clustering.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 1. El Algoritmo K-Means 🧠

**K-Means** es el algoritmo de clustering **basado en representantes (*representative-based*)** más utilizado en la práctica: cada clúster se representa mediante un único punto en el espacio de características, su **centroide** ($\mu_k$), calculado como el promedio de todas las observaciones que pertenecen a ese grupo.

Es un algoritmo **iterativo**: parte de una solución inicial (posiblemente mala) y la va refinando paso a paso hasta que converge. A diferencia de otros métodos que descubren el número de grupos automáticamente, K-Means **requiere conocer de antemano el número de clústeres $K$**.

Formalmente, K-Means busca la partición $\{C_1, C_2, \dots, C_K\}$ que minimiza la **suma de cuadrados intra-cluster** (*Within-Cluster Sum of Squares*, WCSS):

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \lVert x_i - \mu_k \rVert^2$$

### Pseudocódigo

```
1. Elegir el número de clústeres K.
2. Inicializar K centroides.
3. Repetir hasta convergencia:
   a. Asignar cada punto al centroide más cercano.
   b. Actualizar los centroides como el promedio de los puntos asignados a ellos.
4. Retornar los clústeres finales.
```

Cada iteración reduce (o mantiene) el valor de $J$ — el algoritmo converge cuando las asignaciones dejan de cambiar, o se alcanza un número máximo de iteraciones.

---
## Configuración del Entorno de Trabajo 🛠️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request
import warnings
warnings.filterwarnings('ignore')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (8.5, 4.5)
plt.rcParams['font.size'] = 10

def load_dataset(filename, module_folder="10 - Clustering"):
    local_path = os.path.join(os.getcwd(), "data", filename)
    if os.path.exists(local_path):
        return local_path

    parent_path = os.path.join(os.getcwd(), "..", module_folder, "data", filename)
    if os.path.exists(parent_path):
        return parent_path

    raw_url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Science%20programming/{module_folder.replace(' ', '%20')}/data/{filename}"
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    if not os.path.exists(target_path):
        urllib.request.urlretrieve(raw_url, target_path)
    return target_path

print("🚀 Entorno configurado exitosamente para el Módulo 10: Clustering.")

---
## 2. Dataset de Trabajo: Clientes de un Centro Comercial 🛍️

Utilizaremos el dataset `mall_customers.csv` (200 clientes) para ilustrar K-Means con un ejemplo intuitivo y bidimensional: agruparemos clientes según su **ingreso anual** (`Annual_Income_k`, en miles de USD) y su **puntaje de gasto** (`Spending_Score`, de 1 a 100, asignado por el centro comercial según patrones de compra).

In [ ]:
file_mall = load_dataset('mall_customers.csv')
df_mall = pd.read_csv(file_mall)

print(f"Dimensiones del dataset: {df_mall.shape}")
df_mall.head()

In [ ]:
X = df_mall[['Annual_Income_k', 'Spending_Score']].values

plt.figure()
plt.scatter(X[:, 0], X[:, 1], s=30, color="#475569", alpha=0.7)
plt.title("Clientes del Centro Comercial (Sin Agrupar)", fontweight='bold')
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.show()

A simple vista pueden distinguirse aproximadamente 5 grupos bien separados. Antes de aplicar K-Means, **estandarizamos** las variables con `StandardScaler`: aunque ambas columnas están en escalas relativamente comparables aquí, K-Means se basa en distancia Euclidiana, por lo que **cualquier diferencia de escala entre variables distorsiona los clústeres** — la variable con mayor rango dominaría el cálculo de distancias. Estandarizar siempre es una buena práctica por defecto (ver cuaderno 01).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Media tras escalar:", X_scaled.mean(axis=0).round(4))
print("Desv. estándar tras escalar:", X_scaled.std(axis=0).round(4))

---
## 3. Ajustando K-Means con Scikit-Learn ⚡

La clase `sklearn.cluster.KMeans` implementa el algoritmo. Sus parámetros más importantes:

* `n_clusters`: el número $K$ de clústeres a formar.
* `init`: estrategia de inicialización de los centroides (`'k-means++'` por defecto, `'random'`, o un arreglo explícito).
* `n_init`: número de veces que el algoritmo se ejecuta con distintas semillas iniciales, conservando la mejor corrida (menor $J$).
* `random_state`: semilla para reproducibilidad.

Atributos útiles tras el ajuste:

* `cluster_centers_`: las coordenadas de los $K$ centroides finales.
* `labels_`: la etiqueta de clúster asignada a cada observación de entrenamiento.
* `inertia_`: el valor final de $J$ (WCSS) — lo retomaremos en el cuaderno 05 para el método del codo.

In [ ]:
from sklearn.cluster import KMeans

kmeans_model = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
y_predict = kmeans_model.fit_predict(X_scaled)

print("Etiquetas de clúster (primeras 10):", y_predict[:10])
print("Inercia (WCSS) final:", round(kmeans_model.inertia_, 2))
print("Centroides (espacio escalado):")
print(kmeans_model.cluster_centers_.round(3))

In [ ]:
centers_original = scaler.inverse_transform(kmeans_model.cluster_centers_)

plt.figure()
plt.scatter(X[:, 0], X[:, 1], c=y_predict, cmap='viridis', s=30, alpha=0.8)
plt.scatter(
    centers_original[:, 0], centers_original[:, 1],
    c='red', marker='X', s=220, edgecolor='black', linewidth=1.2, label='Centroides'
)
plt.title("K-Means (k=5): Segmentación de Clientes", fontweight='bold')
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.show()

K-Means logra recuperar 5 grupos claramente interpretables en este dataset clásico:

* **Ingreso bajo, gasto bajo:** clientes conservadores.
* **Ingreso bajo, gasto alto:** clientes que gastan por encima de su capacidad — objetivo interesante para campañas de crédito o fidelización.
* **Ingreso medio, gasto medio:** el grupo "promedio", generalmente el más numeroso.
* **Ingreso alto, gasto bajo:** clientes de alto poder adquisitivo pero bajo compromiso — objetivo para campañas de marketing dirigidas.
* **Ingreso alto, gasto alto:** los clientes más rentables — candidatos ideales para programas VIP.

Nótese que **cada observación fue asignada a algún clúster** — a diferencia de DBSCAN (cuaderno 04), K-Means no tiene el concepto de "punto de ruido" u observación que no pertenece a ningún grupo.

---
## 4. Estrategias de Inicialización de Centroides 🎯

El desempeño de K-Means depende fuertemente de la posición inicial de los centroides: un centroide que arranca lejos de los datos reales, o muy cerca de un valor atípico, puede terminar "adueñándose" de un clúster diminuto y sin sentido, mientras el algoritmo queda atrapado en un **óptimo local** pobre. Las estrategias más comunes son:

* **`random`:** seleccionar $K$ puntos al azar del propio dataset como centroides iniciales. Simple, pero propenso a malas inicializaciones.
* **Inicialización basada en clustering jerárquico:** ejecutar clustering jerárquico (cuaderno 03) sobre una muestra pequeña, cortar el dendrograma en $K$ grupos, y usar esos centroides como punto de partida.
* **Inicialización por punto más lejano (*farthest-point*):** comenzar con un punto aleatorio, y repetidamente elegir el punto más alejado de todos los centroides ya seleccionados.
* **`k-means++` (valor por defecto de scikit-learn):** similar a *farthest-point*, pero **probabilística**. Para cada punto $x$ se calcula $D(x)$, la distancia mínima al centroide ya seleccionado más cercano; el siguiente centroide se elige al azar con probabilidad proporcional a $D(x)^2$. Los puntos alejados de los centroides existentes tienen mayor probabilidad de ser elegidos, pero el proceso no es puramente determinista como *farthest-point* — esto lo hace más robusto frente a valores atípicos aislados.

`k-means++` produce, en promedio, inicializaciones considerablemente mejores (menor $J$ final, convergencia más rápida) que la inicialización aleatoria pura, razón por la cual es el valor por defecto de `KMeans(init=...)` en scikit-learn.

In [ ]:
# Comparación: init='random' vs. init='k-means++', variando n_init
results = []
for init_strategy in ['random', 'k-means++']:
    for n_init_val in [1, 10]:
        km = KMeans(n_clusters=5, init=init_strategy, n_init=n_init_val, random_state=42)
        km.fit(X_scaled)
        results.append({
            'init': init_strategy,
            'n_init': n_init_val,
            'inertia': round(km.inertia_, 3)
        })

pd.DataFrame(results)

El parámetro `n_init` controla cuántas veces se repite todo el proceso con distintas semillas de inicialización, conservando la corrida con menor inercia. Con `n_init=1` el resultado depende completamente de una sola inicialización (buena o mala); con `n_init=10` (el valor recomendado y por defecto en versiones recientes de scikit-learn) el riesgo de quedar atrapado en un óptimo local pobre se reduce sustancialmente, a cambio de un costo computacional mayor.

---
## 5. Puntos Clave y Limitaciones de K-Means ⚠️

### Puntos Clave

* K-Means minimiza la suma de cuadrados intra-cluster (WCSS), favoreciendo clústeres **compactos**.
* Al basarse en distancia Euclidiana, es **sensible a la escala de las variables** — siempre estandarizar antes de ajustar.
* Es **sensible a la inicialización** de los centroides — usar `n_init > 1` (o `init='k-means++'`) para mitigar el riesgo de óptimos locales.
* **Todo punto es asignado a algún clúster** — no existe la noción de "no pertenece a ningún grupo" (a diferencia de DBSCAN).

### Limitaciones

K-Means asume implícitamente que los clústeres son **aproximadamente esféricos y de tamaño similar**, porque minimiza distancias Euclidianas al centroide. Esto lo hace **fallar sistemáticamente** en varios escenarios:

* **Formas no esféricas o alargadas:** clústeres con forma de media luna, espiral o anillo no pueden separarse correctamente — K-Means los "cortará" en función de la distancia al centroide, sin importar su forma real.
* **Densidades o tamaños muy distintos:** un clúster grande y disperso puede terminar "robando" puntos de un clúster pequeño y denso cercano.
* **Valores atípicos (*outliers*):** al usar el promedio para calcular centroides, un solo valor atípico extremo puede desplazar significativamente la posición de un centroide.

Estas limitaciones motivan los métodos que veremos en los próximos cuadernos: el **clustering jerárquico** (cuaderno 03) y, especialmente, **DBSCAN** (cuaderno 04), que puede identificar clústeres de forma arbitraria y detectar puntos de ruido explícitamente.

In [ ]:
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.06, random_state=42)
kmeans_moons = KMeans(n_clusters=2, n_init=10, random_state=42)
labels_moons = kmeans_moons.fit_predict(X_moons)

plt.figure()
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels_moons, cmap='viridis', s=25, alpha=0.85)
plt.scatter(
    kmeans_moons.cluster_centers_[:, 0], kmeans_moons.cluster_centers_[:, 1],
    c='red', marker='X', s=200, edgecolor='black', label='Centroides'
)
plt.title("K-Means Falla en Estructuras No Esféricas (make_moons)", fontweight='bold')
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.legend()
plt.show()

El resultado es claro: aunque a simple vista existen dos "medias lunas" perfectamente separables, K-Means las divide con una frontera aproximadamente lineal que pasa por el medio de la figura — porque su criterio de optimización solo entiende de distancia al centroide, no de la forma real de la estructura subyacente.

---
##### 🛠️ Práctica 1: Segmentación de Clientes Incluyendo la Edad

**Objetivo:** Extender el análisis de segmentación de clientes agregando la variable `Age` (edad) al conjunto de características, y comparar el resultado frente al clustering bidimensional original.

**Instrucciones:**
1. Construye una matriz de características `X3 = df_mall[['Age', 'Annual_Income_k', 'Spending_Score']].values`.
2. Estandariza `X3` con un nuevo `StandardScaler`.
3. Ajusta `KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)` sobre los datos escalados.
4. Visualiza el resultado con un `scatter plot` de `Annual_Income_k` vs. `Spending_Score`, coloreando los puntos según la nueva etiqueta de clúster (`c=y_predict3`).
5. Compara: ¿los grupos resultantes son iguales, similares o notablemente distintos a los del clustering bidimensional del apartado 3? ¿Qué podría explicar las diferencias?

In [ ]:
# =========================================================================
# TU SOLUCIÓN: Práctica 1 - Segmentación Incluyendo la Edad
# =========================================================================

# 1. Construir la matriz de características de 3 columnas
# X3 = df_mall[['Age', 'Annual_Income_k', 'Spending_Score']].values

# 2. Estandarizar
# scaler3 = StandardScaler()
# X3_scaled = scaler3.fit_transform(X3)

# 3. Ajustar KMeans
# kmeans3 = KMeans(...)
# y_predict3 = kmeans3.fit_predict(...)

# 4. Visualizar (Annual_Income_k vs. Spending_Score, coloreado por y_predict3)
# plt.figure()
# plt.scatter(...)
# plt.show()

# 5. (Opcional) Compara la inercia de este modelo con la del modelo de 2 variables
# print(kmeans3.inertia_, kmeans_model.inertia_)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1. Construir la matriz de características de 3 columnas
X3 = df_mall[['Age', 'Annual_Income_k', 'Spending_Score']].values

# 2. Estandarizar
scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3)

# 3. Ajustar KMeans
kmeans3 = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
y_predict3 = kmeans3.fit_predict(X3_scaled)

# 4. Visualizar
plt.figure()
plt.scatter(X[:, 0], X[:, 1], c=y_predict3, cmap='viridis', s=30, alpha=0.8)
plt.title("K-Means (k=5) Incluyendo Edad: Ingreso vs. Gasto", fontweight='bold')
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.show()

# 5. Comparar inercia
print(f"Inercia con 3 variables (Age incluida): {kmeans3.inertia_:.2f}")
print(f"Inercia con 2 variables (solo Income/Spending): {kmeans_model.inertia_:.2f}")
```

**Interpretación esperada:** Al incluir `Age`, la partición en el plano Income-Spending suele cambiar de forma sutil — algunos clústeres se "mezclan" parcialmente porque ahora el algoritmo optimiza distancias en un espacio de 3 dimensiones, no solo 2. Las inercias no son directamente comparables entre sí (dimensiones distintas), pero el ejercicio ilustra un punto central: **la elección de qué variables incluir cambia la estructura de los grupos descubiertos** — no existe una única "verdad" de clustering, sino que depende de la pregunta de negocio que se quiera responder.
</details>

---
### 6. Resumen y Conclusiones del Cuaderno 02 📌

1. **Algoritmo Iterativo Basado en Centroides:** K-Means alterna entre asignar puntos al centroide más cercano y recalcular centroides como el promedio de sus puntos asignados, minimizando la suma de cuadrados intra-cluster (WCSS).
2. **La Inicialización Importa:** `k-means++` (probabilística, favorece centroides iniciales alejados entre sí) junto con `n_init > 1` reduce el riesgo de converger a un óptimo local pobre.
3. **Limitaciones Estructurales:** K-Means asume clústeres aproximadamente esféricos y de tamaño similar — falla en estructuras alargadas, anidadas o de densidad variable, y es sensible a valores atípicos.
4. **Próximos Pasos:** El **clustering jerárquico** (cuaderno 03) ofrece una alternativa que no requiere fijar $K$ de antemano y permite explorar la estructura de los datos a múltiples niveles de granularidad mediante un dendrograma.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>